# LeetCode 901 — Online Stock Span bằng Python

Link đề: https://leetcode.com/problems/online-stock-span/ · Độ khó: **Medium** · Chủ đề: Stack, Design, Monotonic Stack, Data Stream

Skeleton LeetCode:

```python
class StockSpanner:

    def __init__(self):
        ...

    def next(self, price: int) -> int:
        ...

# Your StockSpanner object will be instantiated and called as such:
# obj = StockSpanner()
# param_1 = obj.next(price)
```

Ghi chú: bài **thiết kế lớp** xử lý **dòng dữ liệu trực tuyến** (mỗi lần `next` nhận một giá mới). LeetCode chạy một **chuỗi lệnh**; notebook tự dựng bộ chạy `run_ops`. Điểm cốt lõi: **ngăn xếp đơn điệu** lưu `(giá, nhịp)`; khi giá mới `≥` giá ở đỉnh, nhịp của đỉnh được **gộp** vào nhịp mới, nên mỗi giá vào/ra ngăn xếp đúng một lần (chi phí gộp `O(1)`). Oracle: **định nghĩa trực tiếp** — nhịp = khoảng cách tới ngày gần nhất có giá **lớn hơn hẳn**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **nhịp (span)** của một ngày là số ngày liên tiếp tính từ hôm nay ngược về trước mà giá `≤` giá hôm nay.
2. Nắm công cụ Python: lớp có trạng thái, `list` làm ngăn xếp, bộ chạy chuỗi lệnh `run_ops`, vòng `while` pop có điều kiện.
3. Đi từ **lưu hết giá rồi quét ngược (`O(n)` mỗi lần)** → **ngăn xếp `(giá, nhịp)`** → **ngăn xếp `(giá, chỉ số ngày)`** → **mảng nhịp với "nhảy con trỏ"**.
4. Hiểu mấu chốt: nếu giá hôm nay `≥` giá ở đỉnh ngăn xếp thì mọi ngày mà đỉnh đã "phủ" cũng đều `≤` giá hôm nay; ta **gộp nhịp** thay vì quét lại.
5. Tránh lỗi kinh điển: dùng `<` thay vì `≤`, chỉ pop **một lần** (dùng `if` thay `while`), không cộng dồn nhịp đã pop, nhịp khởi tạo sai (`0`), và nhầm giữa "chỉ số" với "số ngày".

## 2. Đề bài & ví dụ

Thiết kế thuật toán thu thập giá cổ phiếu hằng ngày và trả về **nhịp** của giá ngày hiện tại.

**Nhịp** của một ngày là số ngày liên tiếp **tối đa** (bắt đầu từ chính ngày đó, đi ngược về trước) mà giá cổ phiếu **nhỏ hơn hoặc bằng** giá của ngày đó.

Ví dụ: nếu 4 ngày gần nhất có giá `[7,2,1,2]` và hôm nay giá là `2` thì nhịp là `4` (cả 4 ngày đều `≤ 2`). Nếu `[7,34,1,2]` và hôm nay `8` thì nhịp là `3` (`2, 1, 8` rồi gặp `34 > 8`).

Lớp `StockSpanner`: `StockSpanner()` khởi tạo; `int next(int price)` trả nhịp của ngày với giá `price`.

**Ví dụ 1:** các lệnh `next(100), next(80), next(60), next(70), next(60), next(75), next(85)` cho `[1, 1, 1, 2, 1, 4, 6]`.

**Ràng buộc:** `1 <= price <= 10^5`; tối đa `10^4` lệnh `next`.

In [1]:
import random
import time
from typing import List


def run_ops(cls, ops, args):
    """Bộ chạy chuỗi lệnh kiểu LeetCode: ops[0] khởi tạo, các lệnh sau gọi phương thức cùng tên. Trả về danh sách kết quả."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


example_prices = [100, 80, 60, 70, 60, 75, 85]
example_ops = ["StockSpanner"] + ["next"] * 7
example_args = [[]] + [[p] for p in example_prices]
example_out = [None, 1, 1, 1, 2, 1, 4, 6]
print("giá:", example_prices, "-> nhịp:", example_out[1:])

giá: [100, 80, 60, 70, 60, 75, 85] -> nhịp: [1, 1, 1, 2, 1, 4, 6]


## 3. Kiến thức nền

### 3.1 Nhịp và "ngày gần nhất có giá lớn hơn hẳn"

Gọi `a_0, a_1, …` là giá các ngày. Nhịp của ngày `i` là `i − j` với `j` là **chỉ số lớn nhất** `< i` có `a_j > a_i` (nếu không có `j` thì `j = −1`, nhịp `i + 1`). Vì "nhỏ hơn hoặc bằng" nên các ngày có giá **bằng** vẫn được tính vào nhịp.

In [2]:
def span_by_definition(prices):
    out = []
    for i, p in enumerate(prices):
        j = max((k for k in range(i) if prices[k] > p), default=-1)        # ngày gần nhất có giá LỚN HƠN HẲN
        out.append(i - j)
    return out


print(span_by_definition(example_prices), "(khớp ví dụ:", span_by_definition(example_prices) == example_out[1:], ")")

[1, 1, 1, 2, 1, 4, 6] (khớp ví dụ: True )


### 3.2 Ngăn xếp đơn điệu

Ngăn xếp `list`: `append` đẩy, `pop` lấy, `[-1]` xem đỉnh. Ngăn xếp **đơn điệu giảm** giữ các phần tử theo giá giảm dần từ đáy lên đỉnh: trước khi đẩy giá mới ta **pop** mọi phần tử `≤` giá mới (chúng không bao giờ còn là "ngày gần nhất có giá lớn hơn" cho tương lai vì giá mới đã che chúng).

### 3.3 Gộp nhịp (span merging)

Mỗi phần tử trong ngăn xếp lưu `(giá, nhịp)` với nhịp là số ngày mà nó "phủ" (chính nó cộng các ngày bị pop khi nó được đẩy). Khi pop phần tử `(p', s')` do `p' ≤ giá mới`, cộng `s'` vào nhịp của ngày mới: các ngày mà `p'` phủ đều có giá `≤ p' ≤ giá mới`.

### 3.4 Chi phí gộp (amortized)

Mỗi giá được `push` đúng một lần và `pop` tối đa một lần. Tổng chi phí `n` lệnh là `O(n)`, trung bình `O(1)` mỗi lệnh, dù một lệnh đơn lẻ có thể pop nhiều phần tử.

In [3]:
stack = []
for p in [100, 80, 60, 70]:
    span = 1
    while stack and stack[-1][0] <= p:
        span += stack.pop()[1]                         # gộp nhịp của phần tử bị pop
    stack.append((p, span))
    print(f"price {p:3d}: nhịp {span}, ngăn xếp (đáy -> đỉnh) = {stack}")

price 100: nhịp 1, ngăn xếp (đáy -> đỉnh) = [(100, 1)]
price  80: nhịp 1, ngăn xếp (đáy -> đỉnh) = [(100, 1), (80, 1)]
price  60: nhịp 1, ngăn xếp (đáy -> đỉnh) = [(100, 1), (80, 1), (60, 1)]
price  70: nhịp 2, ngăn xếp (đáy -> đỉnh) = [(100, 1), (80, 1), (70, 2)]


## 4. Ý tưởng nền tảng / trực giác

Dãy giá `100, 80, 60, 70, 60, 75, 85`:

```
ngày : 0    1    2    3    4    5    6
giá  : 100  80   60   70   60   75   85
nhịp : 1    1    1    2    1    4    6
```

Ngày `5` (giá `75`): đi ngược: `60 ≤ 75`, `70 ≤ 75`, `60 ≤ 75`, rồi `80 > 75` dừng: nhịp `4` (ngày 2, 3, 4, 5). Thay vì đếm từng ngày, ta nhớ: ngày `4` có nhịp 1, ngày `3` (70) phủ ngày `2` nên nhịp 2; gộp `1 + 2` cộng chính nó `1` ⇒ `4`.

Ngày `6` (giá `85`): ngăn xếp trước đó `[(100,1),(80,5)]`... `80 ≤ 85` pop, nhịp `1 + 5 = 6`; `100 > 85` dừng ⇒ `6`.

In [4]:
hist = []
for p in example_prices:
    back = 0
    for q in reversed(hist + [p]):
        if q <= p:
            back += 1
        else:
            break
    hist.append(p)
    print(f"ngày {len(hist) - 1} giá {p}: đếm ngược các ngày có giá <= {p}: {back}")

ngày 0 giá 100: đếm ngược các ngày có giá <= 100: 1
ngày 1 giá 80: đếm ngược các ngày có giá <= 80: 1
ngày 2 giá 60: đếm ngược các ngày có giá <= 60: 1
ngày 3 giá 70: đếm ngược các ngày có giá <= 70: 2
ngày 4 giá 60: đếm ngược các ngày có giá <= 60: 1
ngày 5 giá 75: đếm ngược các ngày có giá <= 75: 4
ngày 6 giá 85: đếm ngược các ngày có giá <= 85: 6


## 5. Cách 1 — Brute Force: lưu hết giá, quét ngược

**Ý tưởng:** giữ danh sách `prices`. Mỗi `next(price)`: thêm giá, rồi đếm ngược từ ngày hiện tại tới khi gặp giá `> price`.

In [5]:
class SpanBrute:
    def __init__(self):
        self.prices = []

    def next(self, price):
        self.prices.append(price)
        span, i = 0, len(self.prices) - 1
        while i >= 0 and self.prices[i] <= price:       # đếm ngược các ngày giá <= price
            span += 1
            i -= 1
        return span


assert run_ops(SpanBrute, example_ops, example_args) == example_out
print(run_ops(SpanBrute, example_ops, example_args))

[None, 1, 1, 1, 2, 1, 4, 6]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(n) mỗi lệnh: O(n^2) tổng; với n = 10^4 vẫn qua được, nhưng chậm)
class StockSpannerBrute:

    def __init__(self):
        self.prices = []

    def next(self, price: int) -> int:
        self.prices.append(price)
        span, i = 0, len(self.prices) - 1
        while i >= 0 and self.prices[i] <= price:
            span += 1
            i -= 1
        return span

**Độ phức tạp:** `next` `O(n)` trong trường hợp xấu (giá không giảm), tổng `O(n²)`; bộ nhớ `O(n)`.

**Nút thắt:** các ngày bị đếm lại nhiều lần qua các lệnh khác nhau. Một ngày đã "bị phủ" bởi ngày cao hơn trước đó không cần xét lại.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ **ngăn xếp** các ngày "chưa bị che" (giá giảm dần từ đáy lên đỉnh), mỗi phần tử kèm nhịp. Ngày mới:

- pop mọi phần tử có giá `≤` giá mới và **cộng nhịp** của chúng;
- nhịp mới `= 1 + tổng nhịp đã pop`;
- đẩy `(giá mới, nhịp mới)`.

Mỗi phần tử vào/ra đúng một lần ⇒ `O(1)` gộp.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** ngăn xếp `(giá, nhịp)`, **Cách 3** ngăn xếp `(giá, chỉ số ngày)`, **Cách 4** mảng nhịp với "nhảy con trỏ".

### Cách 2 — Ngăn xếp `(giá, nhịp)`

**Giải thích từng đoạn:**

1. `stack = []` các cặp `(giá, nhịp)` với giá giảm dần từ đáy lên đỉnh.
2. `next(price)`: `span = 1`; `while stack and stack[-1][0] <= price: span += stack.pop()[1]`.
3. `stack.append((price, span))`; trả `span`.

In [7]:
class SpanStack:
    def __init__(self):
        self.stack = []                                  # (giá, nhịp), giá giảm dần từ đáy lên đỉnh

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]                  # gộp nhịp của ngày bị che
        self.stack.append((price, span))
        return span


assert run_ops(SpanStack, example_ops, example_args) == example_out
print(run_ops(SpanStack, example_ops, example_args))

[None, 1, 1, 1, 2, 1, 4, 6]


In [8]:
# Code nộp LeetCode cho Cách 2
class StockSpannerStack:

    def __init__(self):
        self.stack = []

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span

**Độ phức tạp:** `next` `O(1)` gộp; bộ nhớ `O(n)` trường hợp xấu (giá giảm dần chặt).

### Cách 3 — Ngăn xếp `(giá, chỉ số ngày)`

**Ý tưởng:** thay vì lưu nhịp, lưu **chỉ số ngày**. Giữ bộ đếm `day`. Sau khi pop các giá `≤ price`, đỉnh còn lại (nếu có) là ngày gần nhất có giá **lớn hơn hẳn**; nhịp `= day − chỉ số đỉnh` (hoặc `day + 1` nếu ngăn xếp rỗng).

In [9]:
class SpanIndex:
    def __init__(self):
        self.day = -1
        self.stack = []                                  # (giá, chỉ số ngày), giá giảm dần

    def next(self, price):
        self.day += 1
        while self.stack and self.stack[-1][0] <= price:
            self.stack.pop()
        span = self.day - (self.stack[-1][1] if self.stack else -1)     # khoảng cách tới ngày gần nhất có giá lớn hơn hẳn
        self.stack.append((price, self.day))
        return span


assert run_ops(SpanIndex, example_ops, example_args) == example_out
print(run_ops(SpanIndex, example_ops, example_args))

[None, 1, 1, 1, 2, 1, 4, 6]


In [10]:
# Code nộp LeetCode cho Cách 3
class StockSpannerIndex:

    def __init__(self):
        self.day = -1
        self.stack = []

    def next(self, price: int) -> int:
        self.day += 1
        while self.stack and self.stack[-1][0] <= price:
            self.stack.pop()
        span = self.day - (self.stack[-1][1] if self.stack else -1)
        self.stack.append((price, self.day))
        return span

**Độ phức tạp:** `O(1)` gộp, bộ nhớ `O(n)`. Dạng này dễ mở rộng cho các bài "ngày gần nhất thoả điều kiện".

### Cách 4 — Mảng nhịp với "nhảy con trỏ"

**Ý tưởng:** không dùng ngăn xếp tường minh. Lưu `prices` và `spans` cho mọi ngày. Với ngày mới `i`, ngày liền trước `i − 1` đã "phủ" `spans[i−1]` ngày, nên ta **nhảy** `j = i − 1`, rồi `j -= spans[j]` chừng nào `prices[j] ≤ price`. Nhịp `= i − j`. Chuỗi `j` đi qua đúng các phần tử của ngăn xếp ẩn.

In [11]:
class SpanJump:
    def __init__(self):
        self.prices, self.spans = [], []

    def next(self, price):
        i = len(self.prices)
        j = i - 1
        while j >= 0 and self.prices[j] <= price:
            j -= self.spans[j]                           # nhảy qua toàn bộ các ngày mà ngày j đã phủ
        span = i - j
        self.prices.append(price)
        self.spans.append(span)
        return span


assert run_ops(SpanJump, example_ops, example_args) == example_out
print(run_ops(SpanJump, example_ops, example_args))

[None, 1, 1, 1, 2, 1, 4, 6]


In [12]:
# Code nộp LeetCode cho Cách 4
class StockSpannerJump:

    def __init__(self):
        self.prices, self.spans = [], []

    def next(self, price: int) -> int:
        i = len(self.prices)
        j = i - 1
        while j >= 0 and self.prices[j] <= price:
            j -= self.spans[j]
        span = i - j
        self.prices.append(price)
        self.spans.append(span)
        return span

**Độ phức tạp:** `O(1)` gộp (mỗi lần nhảy bỏ qua ít nhất một phần tử ngăn xếp ẩn), bộ nhớ `O(n)` (lưu mọi ngày).

## 8. TRACE từng bước

**Trace Cách 2** với `100, 80, 60, 70, 60, 75, 85`:

| giá | các phần tử bị pop `(giá, nhịp)` | nhịp | ngăn xếp sau (đáy → đỉnh) |
|---|---|---|---|
| 100 | — | 1 | `[(100,1)]` |
| 80 | — | 1 | `[(100,1),(80,1)]` |
| 60 | — | 1 | `[(100,1),(80,1),(60,1)]` |
| 70 | `(60,1)` | `1+1 = 2` | `[(100,1),(80,1),(70,2)]` |
| 60 | — | 1 | `[(100,1),(80,1),(70,2),(60,1)]` |
| 75 | `(60,1)`, `(70,2)` | `1+1+2 = 4` | `[(100,1),(80,1),(75,4)]` |
| 85 | `(75,4)`, `(80,1)` | `1+4+1 = 6` | `[(100,1),(85,6)]` |

Hàm trace in đúng các dòng này:

In [13]:
def trace_stack(prices):
    stack = []
    print(f"{'giá':>4}  {'bị pop':<22} {'nhịp':>5}  ngăn xếp sau")
    out = []
    for p in prices:
        span, popped = 1, []
        while stack and stack[-1][0] <= p:
            item = stack.pop()
            popped.append(item)
            span += item[1]
        stack.append((p, span))
        out.append(span)
        print(f"{p:>4}  {str(popped) if popped else '—':<22} {span:>5}  {stack}")
    return out


assert trace_stack(example_prices) == example_out[1:]

 giá  bị pop                  nhịp  ngăn xếp sau
 100  —                          1  [(100, 1)]
  80  —                          1  [(100, 1), (80, 1)]
  60  —                          1  [(100, 1), (80, 1), (60, 1)]
  70  [(60, 1)]                  2  [(100, 1), (80, 1), (70, 2)]
  60  —                          1  [(100, 1), (80, 1), (70, 2), (60, 1)]
  75  [(60, 1), (70, 2)]         4  [(100, 1), (80, 1), (75, 4)]
  85  [(75, 4), (80, 1)]         6  [(100, 1), (85, 6)]


**Trace Cách 4 (nhảy con trỏ)** cho giá `75` ở ngày `5`: `prices = [100,80,60,70,60]`, `spans = [1,1,1,2,1]`. `j = 4` (`60 ≤ 75`) → `j -= spans[4] = 1` → `j = 3` (`70 ≤ 75`) → `j -= spans[3] = 2` → `j = 1` (`80 > 75`) dừng; nhịp `= 5 − 1 = 4` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 2** (ngăn xếp `(giá, nhịp)`). Giữ nguyên tên lớp `StockSpanner`, hàm `next(self, price)`:

In [14]:
class StockSpanner:

    def __init__(self):
        self.stack = []                                   # (giá, nhịp); giá giảm dần từ đáy lên đỉnh

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:  # các ngày có giá <= hôm nay: gộp nhịp của chúng
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


print(run_ops(StockSpanner, example_ops, example_args))      # [None, 1, 1, 1, 2, 1, 4, 6]
assert run_ops(StockSpanner, example_ops, example_args) == example_out

[None, 1, 1, 1, 2, 1, 4, 6]


## 10. Vì sao đúng

**(a) Định nghĩa.** Nhịp của ngày `i` là `i − j`, với `j` là chỉ số lớn nhất `< i` có `a_j > a_i` (hoặc `−1`). Cách 3 tính đúng công thức này: sau khi pop mọi giá `≤ a_i`, đỉnh còn lại là ngày gần nhất có giá `> a_i`, **nếu** bất biến sau đúng.

**(b) Bất biến ngăn xếp.** Trước ngày `i`, ngăn xếp chứa (từ đáy lên đỉnh) chính xác các ngày `j` mà **không có ngày nào sau `j` (đến `i − 1`) có giá `≥ a_j`**, tức các "kỷ lục nghiêm ngặt từ phải sang trái"; giá của chúng **giảm chặt** từ đáy lên đỉnh. Với ngày mới `i`, các ngày `j` trong ngăn xếp với `a_j ≤ a_i` không còn là kỷ lục (bị `i` che) nên bị pop; ngày còn lại có `a_j > a_i` giữ nguyên. Đẩy `i` giữ bất biến.

**(c) Ngày gần nhất có giá `> a_i` nằm trong ngăn xếp.** Giả sử `j*` là ngày gần nhất `< i` có `a_{j*} > a_i`. Mọi ngày sau `j*` (đến `i−1`) có giá `≤ a_i < a_{j*}`, nên `j*` không bị che bởi ngày nào khác trước `i` ⇒ `j*` ∈ ngăn xếp, và nằm ngay **sau** (dưới) các phần tử bị pop vì chúng đều có giá `≤ a_i < a_{j*}`. Vậy sau khi pop, đỉnh là `j*`.

**(d) Cách 2 tương đương Cách 3.** Nhịp lưu của mỗi phần tử ngăn xếp bằng khoảng cách tới phần tử liền dưới nó (bằng quy nạp: nhịp `= 1 +` tổng nhịp các phần tử bị pop = số ngày giữa phần tử dưới đỉnh mới và ngày `i`). Do đó tổng nhịp bị pop cộng `1` chính là `i − j*`.

**(e) Cách 4.** `spans[j]` là nhịp của ngày `j`, tức ngày `j` phủ các ngày `j − spans[j] + 1 … j` toàn giá `≤ a_j`. Nếu `a_j ≤ price` thì cả đoạn đó có giá `≤ price`, nên nhảy tới `j − spans[j]` là đúng. Chuỗi nhảy đi qua đúng các ngày của ngăn xếp ẩn.

**(f) Chi phí gộp.** Mỗi `push` một lần; mỗi `pop` tối đa một lần ⇒ tổng `pop ≤ n`.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **định nghĩa** `i − j*` trên mọi dãy giá nhỏ (mục 12D) và trên dãy dài (12E–F), kèm đếm số `pop` `≤ n`.

## 11. Time / Space Complexity

| Cách | `next` | Space | Ghi chú |
|---|---|---|---|
| 1. Quét ngược | `O(n)` | `O(n)` | oracle thứ hai |
| 2. Ngăn xếp `(giá, nhịp)` | `O(1)` gộp | `O(n)` | **đáp án chính** |
| 3. Ngăn xếp `(giá, chỉ số)` | `O(1)` gộp | `O(n)` | dễ mở rộng |
| 4. Mảng nhịp + nhảy | `O(1)` gộp | `O(n)` | không ngăn xếp tường minh |

Đo thực tế (`10^4` lệnh; chọn dãy xấu nhất cho Cách 1: giá không giảm):

In [15]:
def bench_run(cls, prices, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        obj = cls()
        for p in prices:
            obj.next(p)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(901)
datasets = {
    "ngẫu nhiên": [rng.randint(1, 10 ** 5) for _ in range(10 ** 4)],
    "tăng dần (xấu cho cách 1)": list(range(1, 10 ** 4 + 1)),
    "toàn bằng nhau": [5] * 10 ** 4,
    "giảm dần": list(range(10 ** 4, 0, -1)),
}
for label, prices in datasets.items():
    row = [f"{name} {bench_run(cls, prices) * 1e3:8.2f} ms" for name, cls in
           [("brute", SpanBrute), ("stack", SpanStack), ("index", SpanIndex), ("jump", SpanJump)]]
    print(f"{label:26s}: " + " | ".join(row))

ngẫu nhiên                : brute     6.23 ms | stack     1.61 ms | index     2.15 ms | jump     2.15 ms


tăng dần (xấu cho cách 1) : brute  3982.65 ms | stack     1.30 ms | index     1.72 ms | jump     1.74 ms


toàn bằng nhau            : brute  3071.13 ms | stack     1.22 ms | index     1.61 ms | jump     1.64 ms
giảm dần                  : brute     1.48 ms | stack     1.05 ms | index     1.69 ms | jump     1.24 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề (qua `run_ops`), (B) edge case, (C) phản ví dụ, (D) vét cạn mọi dãy giá nhỏ với oracle định nghĩa, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`10^4` lệnh).

### A. Ví dụ của đề

In [16]:
CLASSES = {
    "brute": SpanBrute,
    "stack": SpanStack,
    "index": SpanIndex,
    "jump": SpanJump,
    "StockSpanner": StockSpanner,
    "StockSpannerBrute": StockSpannerBrute,
    "StockSpannerStack": StockSpannerStack,
    "StockSpannerIndex": StockSpannerIndex,
    "StockSpannerJump": StockSpannerJump,
}


def spans_of(cls, prices):
    obj = cls()
    return [obj.next(p) for p in prices]


def check(prices, names=None):
    want = span_by_definition(prices)                    # oracle: i - (ngày gần nhất có giá lớn hơn hẳn)
    for name in (names or CLASSES):
        got = spans_of(CLASSES[name], prices)
        assert got == want, (name, prices, got, want)
    return want


for cls in CLASSES.values():
    assert run_ops(cls, example_ops, example_args) == example_out
assert check(example_prices) == example_out[1:]
print("ví dụ đề: oracle định nghĩa và", len(CLASSES), "cài đặt (chạy bằng run_ops) đều cho", example_out)

ví dụ đề: oracle định nghĩa và 9 cài đặt (chạy bằng run_ops) đều cho [None, 1, 1, 1, 2, 1, 4, 6]


### B. Edge case

- Một ngày: nhịp `1`.
- Giá bằng nhau liên tục: nhịp `1, 2, 3, …` (bằng nhau vẫn tính).
- Giá giảm chặt: mọi nhịp `1`.
- Giá tăng chặt: nhịp `1, 2, 3, …` (mỗi ngày phủ mọi ngày trước).
- Giá lớn nhất xuất hiện sớm rồi mọi ngày sau nhỏ hơn.
- Răng cưa (lên xuống xen kẽ).
- Giá `1` và `10^5` (biên của ràng buộc).

In [17]:
cases = {
    "một ngày": ([7], [1]),
    "bằng nhau": ([5, 5, 5, 5], [1, 2, 3, 4]),
    "giảm chặt": ([9, 7, 5, 3, 1], [1, 1, 1, 1, 1]),
    "tăng chặt": ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),
    "đỉnh sớm": ([100, 1, 2, 3], [1, 1, 2, 3]),
    "răng cưa": ([3, 1, 3, 1, 3], [1, 1, 3, 1, 5]),
    "biên": ([1, 100000, 1, 100000], [1, 2, 1, 4]),
    "ví dụ đề bài": ([7, 2, 1, 2], [1, 1, 1, 3]),
    "ví dụ thứ hai": ([7, 34, 1, 2, 8], [1, 2, 1, 2, 3]),
}
for name, (prices, want) in cases.items():
    assert check(prices) == want, name
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

9 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
class BugStrict:                                        # pop khi giá đỉnh < giá mới (không pop khi bằng)
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] < price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


class BugIfNotWhile:                                    # chỉ pop một lần
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        if self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


class BugNoAccumulate:                                  # pop nhưng chỉ cộng 1 cho mỗi phần tử, không cộng nhịp đã lưu
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            self.stack.pop()
            span += 1
        self.stack.append((price, span))
        return span


class BugSpanZero:                                      # nhịp khởi tạo 0 (không tính chính ngày hôm nay)
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 0
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


class BugIndexOffByOne:                                 # nhịp = day - chỉ số đỉnh, nhưng day bắt đầu từ 1 mà rỗng dùng -1
    def __init__(self):
        self.day = 0
        self.stack = []

    def next(self, price):
        self.day += 1
        while self.stack and self.stack[-1][0] <= price:
            self.stack.pop()
        span = self.day - (self.stack[-1][1] if self.stack else -1)
        self.stack.append((price, self.day))
        return span


BUGS = [BugStrict, BugIfNotWhile, BugNoAccumulate, BugSpanZero, BugIndexOffByOne]
rng = random.Random(3)
tests = [[rng.randint(1, 6) for _ in range(rng.randint(1, 8))] for _ in range(600)]
for bug in BUGS:
    bad = next((p for p in tests if spans_of(bug, p) != span_by_definition(p)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:17s} sai trên {bad}: ra {spans_of(bug, bad)}, đúng {span_by_definition(bad)}")

BugStrict         sai trên [5, 5, 2, 3]: ra [1, 1, 1, 2], đúng [1, 2, 1, 2]
BugIfNotWhile     sai trên [6, 5, 1, 5, 1, 4, 3, 5]: ra [1, 1, 1, 2, 1, 2, 1, 2], đúng [1, 1, 1, 3, 1, 2, 1, 7]
BugNoAccumulate   sai trên [6, 5, 1, 5, 1, 4, 3, 5]: ra [1, 1, 1, 3, 1, 2, 1, 4], đúng [1, 1, 1, 3, 1, 2, 1, 7]
BugSpanZero       sai trên [5, 5, 2, 3]: ra [0, 0, 0, 0], đúng [1, 2, 1, 2]
BugIndexOffByOne  sai trên [5, 5, 2, 3]: ra [2, 3, 1, 2], đúng [1, 2, 1, 2]


### D. Vét cạn mọi dãy giá nhỏ

Mọi dãy giá độ dài `1..7` với giá `1..4` (`4 + 16 + 64 + 256 + 1 024 + 4 096 + 16 384 = 21 844` dãy) và mọi dãy độ dài `8` với giá `1..3` (`6 561`). Oracle: `i − (ngày gần nhất có giá lớn hơn hẳn)`.

In [19]:
import itertools

t0 = time.perf_counter()
cnt = 0
for n, hi in [(1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 4), (8, 3)]:
    for prices in itertools.product(range(1, hi + 1), repeat=n):
        check(list(prices), names=("brute", "stack", "index", "jump", "StockSpanner"))
        cnt += 1
print(cnt, "dãy giá nhỏ: 5 cài đặt khớp oracle định nghĩa;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 21844 + 6561

28405 dãy giá nhỏ: 5 cài đặt khớp oracle định nghĩa; 0.3 s


### E. Ngẫu nhiên có seed và tính chất

Dãy ngẫu nhiên tới `200` ngày với nhiều dạng (ít giá khác nhau, nhiều giá, tăng/giảm xen kẽ). Tính chất: (1) nhịp luôn trong `[1, i + 1]`; (2) nếu `a_i ≥ a_{i−1}` thì `span_i ≥ span_{i−1} + 1`; (3) kiểm tra trực tiếp theo định nghĩa: mọi ngày trong nhịp có giá `≤ a_i` và ngày ngay trước nhịp (nếu có) có giá `> a_i` (nhịp hợp lệ và tối đại).

In [20]:
rng = random.Random(20240901)
for _ in range(1500):
    n = rng.randint(1, 200)
    mode = rng.choice(["few", "many", "zigzag", "walk"])
    if mode == "few":
        prices = [rng.randint(1, 4) for _ in range(n)]
    elif mode == "many":
        prices = [rng.randint(1, 10 ** 5) for _ in range(n)]
    elif mode == "zigzag":
        prices = [rng.randint(1, 50) if i % 2 else rng.randint(51, 100) for i in range(n)]
    else:
        prices, x = [], 50
        for _ in range(n):
            x = min(10 ** 5, max(1, x + rng.randint(-5, 5)))
            prices.append(x)
    want = check(prices, names=("stack", "index", "jump", "StockSpanner", "StockSpannerIndex", "StockSpannerJump"))
    for i, s in enumerate(want):
        assert 1 <= s <= i + 1                                                    # (1)
        if i > 0 and prices[i] >= prices[i - 1]:
            assert s >= want[i - 1] + 1                                           # (2)
        assert all(prices[k] <= prices[i] for k in range(i - s + 1, i + 1))       # nhịp hợp lệ: mọi ngày trong nhịp có giá <= hôm nay
        assert i - s < 0 or prices[i - s] > prices[i]                             # và ngày ngay trước nhịp có giá lớn hơn (tối đại)
print("1500 dãy ngẫu nhiên: 6 cài đặt khớp oracle; nhịp hợp lệ và tối đại (kiểm tra trực tiếp theo định nghĩa)")

1500 dãy ngẫu nhiên: 6 cài đặt khớp oracle; nhịp hợp lệ và tối đại (kiểm tra trực tiếp theo định nghĩa)


### F. Giới hạn: `10^4` lệnh và đếm số lần `pop`

Chạy `10^4` lệnh trên các dạng dữ liệu xấu; kiểm tra tổng số lần `pop` của Cách 2 `≤ n` (chi phí gộp) và các cách nhanh khớp nhau và khớp Cách 1.

In [21]:
def count_pops(prices):
    stack, pops = [], 0
    for p in prices:
        while stack and stack[-1] <= p:
            stack.pop()
            pops += 1
        stack.append(p)
    return pops


rng = random.Random(500)
n = 10 ** 4
for label, prices in {
    "ngẫu nhiên": [rng.randint(1, 10 ** 5) for _ in range(n)],
    "tăng dần": list(range(1, n + 1)),
    "toàn bằng nhau": [10 ** 5] * n,
    "giảm dần": list(range(n, 0, -1)),
    "răng cưa lớn": [10 ** 5 - (i % 100) * 100 if i % 2 else 1 + i % 100 for i in range(n)],
    "chỉ 2 giá": [rng.choice([1, 10 ** 5]) for _ in range(n)],
}.items():
    ref = spans_of(SpanStack, prices)
    assert spans_of(SpanIndex, prices) == ref == spans_of(SpanJump, prices) == spans_of(StockSpanner, prices)
    if n <= 10 ** 4:
        assert spans_of(SpanBrute, prices) == ref
    pops = count_pops(prices)
    assert pops <= n
    print(f"{label:16s}: max nhịp {max(ref):>5}, số lần pop {pops:>5} <= n = {n}")
print("chi phí gộp: tổng pop <= n trên mọi dạng dữ liệu")

ngẫu nhiên      : max nhịp  3904, số lần pop  9986 <= n = 10000


tăng dần        : max nhịp 10000, số lần pop  9999 <= n = 10000


toàn bằng nhau  : max nhịp 10000, số lần pop  9999 <= n = 10000
giảm dần        : max nhịp     1, số lần pop     0 <= n = 10000
răng cưa lớn    : max nhịp  9902, số lần pop  9950 <= n = 10000


chỉ 2 giá       : max nhịp 10000, số lần pop  9999 <= n = 10000
chi phí gộp: tổng pop <= n trên mọi dạng dữ liệu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `<` thay vì `≤`

Nhịp tính cả các ngày có giá **bằng** hôm nay.

In [22]:
print("giá [5,5,5]:", spans_of(BugStrict, [5, 5, 5]), "| đúng:", spans_of(StockSpanner, [5, 5, 5]))
assert spans_of(BugStrict, [5, 5, 5]) == [1, 1, 1] and spans_of(StockSpanner, [5, 5, 5]) == [1, 2, 3]

giá [5,5,5]: [1, 1, 1] | đúng: [1, 2, 3]


**Vì sao sai:** đề: giá **nhỏ hơn hoặc bằng**. **Đúng:** `while stack and stack[-1][0] <= price`.

### Lỗi 2: `if` thay vì `while`

Chỉ pop một phần tử; các ngày thấp hơn nằm sâu hơn trong ngăn xếp bị bỏ sót.

In [23]:
print("giá [1,2,3,4]:", spans_of(BugIfNotWhile, [1, 2, 3, 4]), "| đúng:", spans_of(StockSpanner, [1, 2, 3, 4]))
assert spans_of(BugIfNotWhile, [1, 2, 3, 4]) == [1, 2, 3, 4]
bad = next(p for p in tests if spans_of(BugIfNotWhile, p) != span_by_definition(p))
print("phản ví dụ:", bad, "->", spans_of(BugIfNotWhile, bad), "| đúng:", span_by_definition(bad))

giá [1,2,3,4]: [1, 2, 3, 4] | đúng: [1, 2, 3, 4]
phản ví dụ: [6, 5, 1, 5, 1, 4, 3, 5] -> [1, 1, 1, 2, 1, 2, 1, 2] | đúng: [1, 1, 1, 3, 1, 2, 1, 7]


**Vì sao sai:** có thể phải pop **nhiều** phần tử mới tới ngày có giá lớn hơn. **Đúng:** vòng `while`.

### Lỗi 3: pop nhưng không cộng nhịp đã lưu

In [24]:
bad = next(p for p in tests if spans_of(BugNoAccumulate, p) != span_by_definition(p))
print(bad, "->", spans_of(BugNoAccumulate, bad), "| đúng:", span_by_definition(bad))

[6, 5, 1, 5, 1, 4, 3, 5] -> [1, 1, 1, 3, 1, 2, 1, 4] | đúng: [1, 1, 1, 3, 1, 2, 1, 7]


**Vì sao sai:** phần tử bị pop đã đại diện cho **nhiều** ngày (`span` của nó), không chỉ một. **Đúng:** `span += stack.pop()[1]`.

### Lỗi 4: nhịp khởi tạo `0`

In [25]:
print("giá [3]:", spans_of(BugSpanZero, [3]), "| đúng:", spans_of(StockSpanner, [3]))
assert spans_of(BugSpanZero, [3]) == [0]

giá [3]: [0] | đúng: [1]


**Vì sao sai:** chính hôm nay cũng được tính vào nhịp (`≥ 1`). **Đúng:** `span = 1`.

### Lỗi 5: lệch chỉ số ngày (Cách 3)

In [26]:
print("giá [3, 1]:", spans_of(BugIndexOffByOne, [3, 1]), "| đúng:", spans_of(StockSpannerIndex, [3, 1]))
assert spans_of(BugIndexOffByOne, [3, 1]) != spans_of(StockSpannerIndex, [3, 1])

giá [3, 1]: [2, 1] | đúng: [1, 1]


**Vì sao sai:** nếu `day` bắt đầu từ `1` thì "ngăn xếp rỗng" phải ứng với chỉ số `0`, không phải `-1`. **Đúng:** thống nhất cách đánh chỉ số (`day` từ `0` và rỗng ↔ `-1`).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "với mỗi phần tử, tìm phần tử **gần nhất** phía trước thoả quan hệ lớn/nhỏ hơn" hoặc "độ dài đoạn liên tiếp phía trước có giá ≤" → **ngăn xếp đơn điệu**. Dạng trực tuyến (dòng dữ liệu) thích hợp vì ngăn xếp được cập nhật dần.

**Mẹo:**

1. Quyết định ngăn xếp giảm hay tăng theo quan hệ (ở đây: pop khi `≤`, ngăn xếp giá giảm chặt).
2. Lưu thông tin đã gộp (`nhịp`) hoặc chỉ số ngày để suy ra khoảng cách.
3. Nhớ vòng `while`, không phải `if`.
4. Ở bài này `≤` (bằng nhau cùng nhịp); ở các bài khác có thể cần `<` — đọc đề kỹ.
5. Oracle: khoảng cách tới ngày gần nhất có giá **lớn hơn hẳn**.

**Họ bài cùng pattern:** Daily Temperatures (739), Next Greater Element I/II (496, 503), Largest Rectangle in Histogram (84), Sum of Subarray Minimums (907), Maximum Width Ramp (962), Stock Price Fluctuation (2034).

## 15. Biến thể

1. **Nhịp "nghiêm ngặt"** (`<` thay vì `≤`): ngăn xếp pop khi `<`.
2. **Daily Temperatures (739):** số ngày phải chờ để gặp ngày ấm hơn (hướng tới tương lai, xử lý ngược).
3. **Nhịp về phía sau** (số ngày liên tiếp phía sau có giá ≤).
4. **Truy vấn ngày bất kỳ** (không chỉ ngày mới nhất): offline với mảng `prev_greater`.
5. **Nhịp lớn nhất đã gặp.**

Cài biến thể 1, 2 và đối chiếu oracle:

In [27]:
class StrictSpanner:
    """Biến thể 1: nhịp nghiêm ngặt = số ngày liên tiếp (tính cả hôm nay) có giá < hôm nay, cộng chính hôm nay (nên ngày bằng giá chặn lại)."""
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] < price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


def strict_by_definition(prices):
    out = []
    for i, p in enumerate(prices):
        j = max((k for k in range(i) if prices[k] >= p), default=-1)       # ngày gần nhất có giá >= hôm nay chặn lại
        out.append(i - j)
    return out


def daily_temperatures(temps):
    ans, stack = [0] * len(temps), []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            ans[j] = i - j
        stack.append(i)
    return ans


def daily_temperatures_brute(temps):
    n = len(temps)
    return [next((j - i for j in range(i + 1, n) if temps[j] > temps[i]), 0) for i in range(n)]


rng = random.Random(15)
for _ in range(500):
    prices = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    assert spans_of(StrictSpanner, prices) == strict_by_definition(prices)
    assert daily_temperatures(prices) == daily_temperatures_brute(prices)
assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
print("nhịp nghiêm ngặt và Daily Temperatures (739) khớp oracle trên 500 dãy")

nhịp nghiêm ngặt và Daily Temperatures (739) khớp oracle trên 500 dãy


## 16. So sánh các cách

| Cách | Ý tưởng | `next` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét ngược | đếm từng ngày | `O(n)` | `O(n)` | oracle thứ hai |
| 2. Ngăn xếp `(giá, nhịp)` | gộp nhịp khi pop | `O(1)` gộp | `O(n)` | **đáp án chính** |
| 3. Ngăn xếp `(giá, chỉ số)` | nhịp = khoảng cách chỉ số | `O(1)` gộp | `O(n)` | dễ mở rộng |
| 4. Mảng nhịp + nhảy | `j -= spans[j]` | `O(1)` gộp | `O(n)` | không ngăn xếp tường minh |

Cả bốn khớp oracle định nghĩa trên 28 405 dãy nhỏ và các dãy tới `10^4` ngày (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [28]:
print(spans_of(StockSpanner, [10, 4, 5, 90, 120, 80]), spans_of(StockSpanner, [2, 2, 1, 3]), spans_of(StrictSpanner, [2, 2, 1, 3]))

[1, 1, 2, 4, 5, 1] [1, 2, 1, 4] [1, 1, 1, 4]


### Bài 2 (tính tay)

Giá `[31, 41, 48, 59, 79]` rồi thêm `[35, 70]`. (a) Nhịp từng ngày? (b) Ngăn xếp `(giá, nhịp)` sau mỗi ngày? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện ngăn xếp `(giá, nhịp)`:

In [29]:
class SpanTodo:
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            pass        # TODO 1: pop và cộng nhịp của phần tử bị pop vào span
        pass            # TODO 2: đẩy (price, span)
        return span


# Khi làm xong, bỏ comment để kiểm tra:
# assert run_ops(SpanTodo, example_ops, example_args) == example_out

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySpanner` bằng **mảng nhịp + nhảy con trỏ**:

In [30]:
class MySpanner:
    def __init__(self):
        pass            # TODO

    def next(self, price):
        pass            # TODO


# Khi làm xong:
# assert run_ops(MySpanner, example_ops, example_args) == example_out

### Bài 5 (mini challenge): nhịp nhỏ hơn về phía sau (offline)

Cho mảng `prices`, với mỗi chỉ số `i` tính `forward_span[i]` = số ngày liên tiếp **bắt đầu từ `i` đi về phía sau** (chỉ số tăng) mà giá `≤ prices[i]`. Viết `forward_spans(prices)` bằng ngăn xếp duyệt từ phải sang trái, kiểm tra bằng vét cạn.

In [31]:
def forward_spans(prices):
    pass            # TODO


# Khi làm xong:
# assert forward_spans([100, 80, 60, 70, 60, 75, 85]) == [7, 5, 1, 2, 1, 1, 1]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ngăn xếp pop khi `≤`.
- Mức 2: `[10,4,5,90,120,80]`: nhịp `1,1,2,4,5,1`; `[2,2,1,3]`: `1,2,1,4`.
- Mức 3: nghiêm ngặt `[2,2,1,3]`: `1,1,1,4` (ngày 1 giá `2` không pop được `2` ở ngày 0 vì không `<`; ngày 3 giá `3` pop cả `1` lẫn hai số `2` nên nhịp `4`).

**Bài 2**

- Mức 1: giá tăng dần `31,41,48,59,79` ⇒ nhịp `1,2,3,4,5`.
- Mức 2: `35`: pop không (`79 > 35`) ⇒ nhịp `1`; `70`: pop `35` (nhịp 1) ⇒ `2`; `79 > 70` dừng.
- Mức 3: nhịp `1,2,3,4,5,1,2`; ngăn xếp cuối `[(79,5),(70,2)]`.

**Bài 3**

- Mức 1: `item = self.stack.pop()`.
- Mức 2: `span += item[1]`.
- Mức 3: `self.stack.append((price, span))`.

**Bài 4**

- Mức 1: lưu `prices` và `spans`.
- Mức 2: `j = i - 1; while j >= 0 and prices[j] <= price: j -= spans[j]`.
- Mức 3: `span = i - j`.

**Bài 5**

- Mức 1: duyệt `i` từ phải sang trái với ngăn xếp các chỉ số.
- Mức 2: pop khi `prices[top] <= prices[i]`.
- Mức 3: `forward_span[i] = (chỉ số đỉnh còn lại hoặc n) - i`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [32]:
assert spans_of(StockSpanner, [10, 4, 5, 90, 120, 80]) == [1, 1, 2, 4, 5, 1] == span_by_definition([10, 4, 5, 90, 120, 80])
assert spans_of(StockSpanner, [2, 2, 1, 3]) == [1, 2, 1, 4]
assert spans_of(StrictSpanner, [2, 2, 1, 3]) == [1, 1, 1, 4] == strict_by_definition([2, 2, 1, 3])
print("Bài 1 OK")

Bài 1 OK


### Bài 2

In [33]:
prices = [31, 41, 48, 59, 79, 35, 70]
st, out, snaps = [], [], []
for p in prices:
    span = 1
    while st and st[-1][0] <= p:
        span += st.pop()[1]
    st.append((p, span))
    out.append(span)
    snaps.append(list(st))
assert out == [1, 2, 3, 4, 5, 1, 2] == span_by_definition(prices)
assert snaps[-1] == [(79, 5), (70, 2)]
print("Bài 2 OK:", out, snaps[-1])

Bài 2 OK: [1, 2, 3, 4, 5, 1, 2] [(79, 5), (70, 2)]


### Bài 3

In [34]:
class SpanAnswer:
    def __init__(self):
        self.stack = []

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


assert run_ops(SpanAnswer, example_ops, example_args) == example_out
for p in tests[:300]:
    assert spans_of(SpanAnswer, p) == span_by_definition(p)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [35]:
class MySpannerAnswer:
    def __init__(self):
        self.prices, self.spans = [], []

    def next(self, price):
        i = len(self.prices)
        j = i - 1
        while j >= 0 and self.prices[j] <= price:
            j -= self.spans[j]
        self.prices.append(price)
        self.spans.append(i - j)
        return i - j


assert run_ops(MySpannerAnswer, example_ops, example_args) == example_out
for p in tests[:300]:
    assert spans_of(MySpannerAnswer, p) == span_by_definition(p)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [36]:
def forward_spans_answer(prices):
    n = len(prices)
    out, stack = [0] * n, []
    for i in range(n - 1, -1, -1):
        while stack and prices[stack[-1]] <= prices[i]:
            stack.pop()
        out[i] = (stack[-1] if stack else n) - i                 # tới ngày kế tiếp có giá lớn hơn hẳn (hoặc hết mảng)
        stack.append(i)
    return out


def forward_spans_brute(prices):
    n = len(prices)
    res = []
    for i in range(n):
        k = i
        while k < n and prices[k] <= prices[i]:
            k += 1
        res.append(k - i)
    return res


assert forward_spans_answer([100, 80, 60, 70, 60, 75, 85]) == [7, 5, 1, 2, 1, 1, 1]
rng = random.Random(5)
for _ in range(500):
    p = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    assert forward_spans_answer(p) == forward_spans_brute(p)
print("Bài 5 OK:", forward_spans_answer([100, 80, 60, 70, 60, 75, 85]))

Bài 5 OK: [7, 5, 1, 2, 1, 1, 1]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (nhịp trực tuyến → ngăn xếp đơn điệu `(giá, nhịp)`):**

```python
class StockSpanner:
    def __init__(self):
        self.stack = []                              # (giá, nhịp), giá giảm dần

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]              # gộp nhịp
        self.stack.append((price, span))
        return span
```

**Checklist trước khi submit:**

- [ ] Pop khi `giá đỉnh <= price` (bằng nhau cũng gộp)?
- [ ] Dùng `while`, không `if`?
- [ ] `span = 1` ban đầu, cộng **nhịp** của phần tử bị pop (không phải `1`)?
- [ ] Đẩy `(price, span)` sau khi pop?
- [ ] Đã thử `[100,80,60,70,60,75,85] → [1,1,1,2,1,4,6]`, giá bằng nhau, tăng dần, giảm dần?
- [ ] Đúng tên `class StockSpanner`, `next(self, price)`, không in debug.